# Importing Libraries

In [2]:
from umap import UMAP # for UMAP latent space projections
import sys # for relatove imports of sigma

Using relative imports for sigma and its packages whilst they are being redeveloped

In [3]:
sys.path.insert(0,"..")
from sigma.utils.base import *
from sigma.utils import normalisation as norm 
from sigma.utils import visualisation as visual

from sigma.src.utils import same_seeds
from sigma.src.dim_reduction import Experiment
from sigma.models.autoencoder import AutoEncoder
from sigma.src.segmentation import PixelSegmenter
from sigma.gui import gui
from sigma.utils.loadtem import TEMDataset
from sigma.utils.load import *

## Loading the data from a `.rpl` file

Details on how to export the required files from aztec can be found in the `guides` folder


In [4]:
file_path='ryugu_aztec_data/EDS444.rpl'
#bse_img_path='ryugu_aztec_data/BSE444.tif' #pointing to the bse image used for navigation / overlays
#file_path='23004_188/site 7/EDS Data.rpl'
sem=SEMDataset(file_path)#,nag_file_path=bse_img_path)

could not find EMSA/MAS file assuming scale of 0.01keV /Ch and offset of -0.2 keV
reading x/y scaling parameters from ryugu_aztec_data/EDS444.par
Unable to read X-Ray lines from sample metadata, setting blank list


In [7]:
sem.base_dataset.axes_manager[2].offset

-0.2

# Calibration of the energy axis - optional

If no spectrum file is recorded - It is neccesary to apply a linear offset correction to the spectral axis of the form:
$E_{corrected}​=a⋅E_{measured}​+b$

This is done automatically with the `calibrate_spectra` dataset.

It will attempt to calibrate automatically, but when there are many peaks in a similar region, it is more robust to provide a dictionary of measured values. These measured peak energies can be found by inspecting the sum spectra with `gui.view_dataset`

In [8]:
gui.view_dataset(sem)

Output()

Output()

<Figure size 0x300 with 0 Axes>

In [22]:
measured_peaks={'O_Ka':0.703,'Fe_Ka':6.44,'S_Ka':2.45}

In [23]:
sem.calibrate_spectra(measured_peaks_dict=measured_peaks)

[Manual] O_Ka: Measured = 0.703 keV → Ref = 0.525 keV
[Manual] Fe_Ka: Measured = 6.440 keV → Ref = 6.404 keV
[Manual] S_Ka: Measured = 2.450 keV → Ref = 2.308 keV
Calibration: E_corrected = 1.025065 * E_measured + -0.198815
Calibration successful.


In [20]:
sem.rebin_signal(size=(3,3)) # rebinning to improve quality of elemental maps

Rebinning the intensity with the size of (3, 3)


(<EDSSEMSpectrum, title: , dimensions: (170, 117|2048)>,
 <Signal2D, title: , dimensions: (|170, 117)>)

In [24]:
gui.view_dataset(sem)

Output()

Output()

# Processing the dataset ahead of projection and clustering

In [14]:
sem.normalisation([norm.neighbour_averaging,norm.zscore])

Set feature_list to ['Fe_Ka', 'Mg_Ka', 'O_Ka', 'S_Ka', 'Si_Ka', 'P_Ka', 'Ni_Ka', 'Na_Ka', 'Al_Ka', 'Ca_Ka']
Normalise dataset using:
    1. neighbour_averaging
    2. zscore


In [7]:
gui.view_pixel_distributions(sem,norm_list=[norm.neighbour_averaging,norm.zscore,norm.softmax],cmap='Reds')

# Projection with UMAP

In [15]:
data = sem.normalised_elemental_data.reshape(-1,len(sem.feature_list))
umap = UMAP(
        n_neighbors=20,
        min_dist=0.05,
        n_components=2,
        metric='euclidean'
    )
latent = umap.fit_transform(data)

# Clustering with GMM

In [28]:
ps_gmm=PixelSegmenter(latent=latent,dataset=sem,method='GaussianMixture',method_args={'n_components' :50, 'random_state':6, 'init_params':'kmeans'} )

## Clustering with HDBSCAN

In [ ]:
ps_hdb = PixelSegmenter(latent=latent, 
                    dataset=sem,
                    method="HDBSCAN",
                    method_args=dict(min_cluster_size=10, min_samples=10,
                                     max_cluster_size=int(len(latent)/10),
                                     cluster_selection_epsilon=1e-1) )

# Plotting the clusters interactively

In [ ]:
gui.interactive_latent_plot(ps=ps_hdb,ratio_to_be_shown=1.0)

In [ ]:
gui.check_latent_space(ps=ps_hdb,show_map=True,ratio_to_be_shown=1.0)

In [32]:
gui.view_clusters_sum_spectra(ps=ps_hdb)

SelectMultiple(options=('cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster…

Output()

In [34]:
gui.view_phase_map(ps=ps_hdb,alpha_cluster_map=0.4)

In [35]:
gui.show_cluster_distribution(ps=ps_hdb)

Output()

# NMF

In [36]:
weights, components = ps_hdb.get_unmixed_spectra_profile(clusters_to_be_calculated='All', 
                                                 n_components=3,
                                                 normalised=False, 
                                                 method='NMF', 
                                                 method_args={'init':'nndsvd'})

In [37]:
gui.show_unmixed_weights_and_compoments(ps=ps_hdb, weights=weights, components=components)

In [38]:
gui.show_abundance_map(ps=ps_hdb, weights=weights, components=components)

Output()

In [1]:
import hyperspy.api as hs

In [3]:
s_h5=hs.load("aztec_data/Will Test 12_LK_35_tests for EDS dwell times LK35_2000msEDS Site 1 Map Data 50.h5oina")

WARNING | Hyperspy | Unable to infer file type from extension 'h5usid'. Will attempt to load the file with the Python imaging library. (hyperspy.io:91)
ERROR | Hyperspy | If this file format is supported, please report this error to the RosettaSciIO developers at https://github.com/hyperspy/rosettasciio/issues (hyperspy.io:600)


OSError: cannot find loader for this HDF5 file

In [4]:
hs.__version__

'2.3.0'